# 🕵️‍♀️ Pipeline Master Job Catalog (JobStreet Scraping)

Notebook ini dibuat untuk melakukan penarikan data (*scraping*) lowongan pekerjaan IT dari JobStreet. 
Proses dibagi menjadi 4 tahap utama:
1. Setup bot browser (Selenium).
2. Penarikan daftar *link* lowongan dari halaman utama.
3. Penarikan detail deskripsi dari masing-masing *link*.
4. Ekstraksi entitas skill IT dan penyimpanan data ke format CSV.

## 1. Setup Browser & Library

In [1]:
import pandas as pd
import time
import random
import re
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from IPython.display import display

print("Menyiapkan Browser Virtual...")

# Konfigurasi Anti-Blokir
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

# Membuka Chrome
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)
BASE_URL = "https://id.jobstreet.com/id/jobs-in-information-communication-technology"

print("✅ Browser siap digunakan!")

Menyiapkan Browser Virtual...
✅ Browser siap digunakan!


## 2. Definisi Kamus Skill
Tahap ini berfungsi menyiapkan *Knowledge Base* sederhana untuk mendeteksi *skill* IT dari paragraf panjang menggunakan *Regular Expression* (RegEx).

In [6]:
# Daftar skill IT yang akan dicari oleh AI
KAMUS_SKILL = [
    # --- Bahasa Pemrograman Dasar & Spesifik ---
    "Python", "SQL", "Java", "C++", "C#", "JavaScript", "TypeScript", 
    "HTML", "CSS", "PHP", "Go", "Golang", "Ruby", "Kotlin", "Swift", "Dart", "R", "Prolog",
    
    # --- Frontend, Mobile & UI/UX ---
    "React", "ReactJS", "Vue", "Angular", "Next.js", "Tailwind", 
    "Flutter", "React Native", "Figma", "Sketch", "Adobe XD", "UI/UX", "Wireframing", 
    "Prototyping", "User Research", "User Journey Map", "Design Sprint",
    
    # --- Backend & Framework ---
    "Node.js", "NodeJS", "Express", "Laravel", "Spring", "Django", "Flask", 
    ".NET", "Ruby on Rails", "FastAPI",
    
    # --- Cloud, DevOps & Network (Dari artikel Network Engineer) ---
    "Linux", "AWS", "Azure", "GCP", "Docker", "Kubernetes", "Git", "GitHub", 
    "GitLab", "CI/CD", "Jenkins", "Networking", "Cisco", "Routing", "Switching", "CCNA",
    
    # --- Database & API ---
    "MySQL", "PostgreSQL", "MongoDB", "Redis", "Oracle", "DDL",
    "API", "REST API", "GraphQL", "Postman", "JSON",
    
    # --- Data, AI & Machine Learning (Dari artikel AI & Data Analytics) ---
    "Machine Learning", "Deep Learning", "NLP", "Data Analysis", "Data Engineering",
    "Tableau", "Power BI", "Excel", "Pandas", "TensorFlow", "PyTorch", "Hadoop", "Spark",
    "Generative AI", "GenAI", "LLM", "Prompt Engineering", "AHP", "SAW",
    
    # --- Cyber Security (Dari artikel Roadmap Cyber Security) ---
    "Cyber Security", "Penetration Testing", "Kali Linux", "Cryptography",
    
    # --- Game Development (Dari artikel Game Design) ---
    "Unity", "Unreal Engine",
    
    # --- Metodologi & Soft Skill Teknis ---
    "Agile", "Scrum", "Troubleshooting", "QA", "Software Testing"
]

def ekstrak_skill(teks):
    """Fungsi pembaca teks untuk mencari skill yang cocok dengan kamus."""
    if not isinstance(teks, str) or teks in ["Deskripsi tidak ditemukan", "Error"]:
        return "[]"
    
    # Mencari kata yang sama persis (\b)
    skill_ditemukan = [skill for skill in KAMUS_SKILL if re.search(r'\b' + re.escape(skill.lower()) + r'\b', teks.lower())]
    
    # Mengembalikan format string array: '["Skill 1", "Skill 2"]'
    if skill_ditemukan:
        return '["' + '", "'.join(skill_ditemukan) + '"]'
    else:
        return "[]"

print(f"✅ Kamus berhasil dimuat dengan {len(KAMUS_SKILL)} skill!")

✅ Kamus berhasil dimuat dengan 102 skill!


## 3. Scraping Halaman Utama (Mengambil Link)
Bot akan membuka halaman pencarian dan menyalin semua judul, nama perusahaan, lokasi, dan yang terpenting: **Job URL**. Parameter `START_PAGE` dan `END_PAGE` bisa diatur untuk mencicil pekerjaan (Batching).

In [34]:
# Tentukan rentang halaman yang mau di-scrape (contoh: hal 1 sampai 10)
START_PAGE = 91
END_PAGE = 100

all_jobs = []
print(f"Memulai Scraping Halaman {START_PAGE} sampai {END_PAGE}...")

for page in range(START_PAGE, END_PAGE + 1):
    print(f"    ⏳ Mengambil data halaman {page}...")
    try:
        driver.get(f"{BASE_URL}?page={page}")
        time.sleep(random.uniform(4, 7)) # Jeda agar halaman ter-render
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        job_cards = soup.find_all('article')
        
        for card in job_cards:
            job_data = {}
            title_elem = card.find('a', {'data-automation': 'jobTitle'})
            
            if title_elem:
                job_data['Job Title'] = title_elem.text.strip()
                link = title_elem.get('href', '')
                job_data['Job URL'] = f"https://id.jobstreet.com{link}" if link.startswith('/') else link
                
                comp_elem = card.find('a', {'data-automation': 'jobCompany'})
                job_data['Company'] = comp_elem.text.strip() if comp_elem else "Perusahaan Dirahasiakan"
                
                loc_elem = card.find('a', {'data-automation': 'jobLocation'})
                job_data['Location'] = loc_elem.text.strip() if loc_elem else None
                
                all_jobs.append(job_data)
    except Exception as e:
        print(f"    [!] Error di halaman {page}: {e}")

df_jobs = pd.DataFrame(all_jobs)
print(f"\n✅ Selesai! Berhasil mengamankan {len(df_jobs)} link loker.")
display(df_jobs.head(3))

Memulai Scraping Halaman 91 sampai 100...
    ⏳ Mengambil data halaman 91...
    ⏳ Mengambil data halaman 92...
    ⏳ Mengambil data halaman 93...
    ⏳ Mengambil data halaman 94...
    ⏳ Mengambil data halaman 95...
    ⏳ Mengambil data halaman 96...
    ⏳ Mengambil data halaman 97...
    ⏳ Mengambil data halaman 98...
    ⏳ Mengambil data halaman 99...
    ⏳ Mengambil data halaman 100...

✅ Selesai! Berhasil mengamankan 145 link loker.


,Job Title,Job URL,Company,Location
0,IT Security & GRC (Lead/Manager),https://id.jobstreet.com/id/job/91746053?type=...,PT Dwi Cermat Indonesia,Jakarta Raya
1,DevOps / Cloud Engineer,https://id.jobstreet.com/id/job/91576176?type=...,PT Simplify Consulting Indonesia,Jakarta Raya
2,Lowongan Kerja Senior .Net Engineer,https://id.jobstreet.com/id/job/89100232?type=...,PT Berlian Sistem Informasi,Jakarta Raya


## 4. Ekstraksi Job Description & Skill IT
Bot akan masuk ke masing-masing *link* yang didapat dari Tahap 3. Ia akan mengambil paragraf *Job Description*, lalu secara otomatis menyaringnya menggunakan fungsi `ekstrak_skill` yang dibuat di Tahap 2.

In [35]:
print("Membuka URL untuk mengambil Job Description mentah...\n")

deskripsi_list = []
skill_list = []

for i, url in enumerate(df_jobs['Job URL']):
    print(f"    ⏳ Memproses loker {i+1} dari {len(df_jobs)}...", end="\r") 
    try:
        driver.get(url)
        time.sleep(random.uniform(3, 6)) # Jeda agak cepat
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Mengincar tag yang sering dipakai JobStreet untuk deskripsi
        desc_elem = soup.find(attrs={'data-automation': 'jobAdDetails'}) or soup.find(attrs={'data-automation': 'jobDescription'}) or soup.find(attrs={'data-automation': 'job-details'})
        
        if desc_elem:
            teks_bersih = desc_elem.get_text(separator=' | ', strip=True)
            deskripsi_list.append(teks_bersih)
            skill_list.append(ekstrak_skill(teks_bersih))
        else:
            deskripsi_list.append("Deskripsi tidak ditemukan")
            skill_list.append("[]")
            
    except Exception as e:
        deskripsi_list.append("Error")
        skill_list.append("[]")

# Memasukkan array ke dalam tabel pandas
df_jobs['Job Description'] = deskripsi_list
df_jobs['required_skills'] = skill_list

print(f"\n\n✅ Pengambilan detail selesai!")

Membuka URL untuk mengambil Job Description mentah...

    ⏳ Memproses loker 145 dari 145...

✅ Pengambilan detail selesai!


## 5. Export Raw Data ke CSV
Menyimpan *DataFrame* utuh tanpa membuang baris mana pun, sebagai *checkpoint* data mentah.

In [36]:
print("Menyimpan Raw Data ke CSV...")

# Penamaan dinamis berdasarkan halaman yang diambil
nama_file = f'raw_job_catalog_page_{START_PAGE}_to_{END_PAGE}.csv'

# Export ke CSV (index=False agar nomor urut bawaan pandas tidak ikut tersimpan)
df_jobs.to_csv(nama_file, index=False)

print(f"🎉 SELESAI! Data mentah berhasil diamankan ke dalam file: '{nama_file}'")

# Menampilkan 5 data teratas untuk memastikan strukturnya masuk akal
display(df_jobs.head())

Menyimpan Raw Data ke CSV...
🎉 SELESAI! Data mentah berhasil diamankan ke dalam file: 'raw_job_catalog_page_91_to_100.csv'


,Job Title,Job URL,Company,Location,Job Description,required_skills
0,IT Security & GRC (Lead/Manager),https://id.jobstreet.com/id/job/91746053?type=...,PT Dwi Cermat Indonesia,Jakarta Raya,Company Description | Cermati is a financial t...,[]
1,DevOps / Cloud Engineer,https://id.jobstreet.com/id/job/91576176?type=...,PT Simplify Consulting Indonesia,Jakarta Raya,About the Role | You will operate the global b...,"[""AWS"", ""Azure"", ""Docker"", ""Kubernetes"", ""CI/CD""]"
2,Lowongan Kerja Senior .Net Engineer,https://id.jobstreet.com/id/job/89100232?type=...,PT Berlian Sistem Informasi,Jakarta Raya,Info Terbaru Lowongan Kerja Senior Net Enginee...,"[""SQL"", ""Java"", ""JavaScript"", ""AWS"", ""Azure"", ..."
3,Application Developer (.Net & Angular Js),https://id.jobstreet.com/id/job/91425191?type=...,PT. SIGMA GLOBAL TEKNOLOGI (SIGMATECH),Jakarta Raya,Notes | Status: PKWT 3 months under Sigmatech...,"[""Python"", ""SQL"", ""JavaScript"", ""TypeScript"", ..."
4,[165] Senior PHP Developer,https://id.jobstreet.com/id/job/87867558?type=...,Granitor/CheckProof,Bandung,We are looking for a | skilled Senior PHP Deve...,"[""SQL"", ""TypeScript"", ""PHP"", ""Angular"", ""Larav..."


## 6. Menggabungkan Semua File CSV (Finalisasi Batching)
Tahap ini dijalankan **HANYA JIKA** semua proses batching dari berbagai halaman sudah selesai. Skrip ini akan secara otomatis mendeteksi semua file berawalan `raw_job_catalog_page_` di dalam folder, membacanya, lalu menggabungkannya (`concat`) menjadi satu file `MASTER_RAW_JOB_CATALOG.csv`.

In [37]:
import glob

print("[*] Memulai proses penggabungan file...")

# Mencari semua file yang namanya cocok dengan pola kita
semua_file_csv = glob.glob("raw_job_catalog_page_*.csv")

if len(semua_file_csv) == 0:
    print("❌ Tidak ada file CSV yang ditemukan untuk digabungkan.")
else:
    print(f"✅ Ditemukan {len(semua_file_csv)} file untuk digabungkan:")
    for f in semua_file_csv:
        print(f"   - {f}")
    
    # Membaca dan menggabungkan semua file ke dalam satu DataFrame
    list_df = [pd.read_csv(file) for file in semua_file_csv]
    df_gabungan = pd.concat(list_df, ignore_index=True)
    
    # Menyimpan file Master
    nama_file_master = "MASTER_RAW_JOB_CATALOG.csv"
    df_gabungan.to_csv(nama_file_master, index=False)
    
    print("\n🎉 PENGGABUNGAN SUKSES!")
    print(f"Total keseluruhan data yang berhasil kamu kumpulkan: {len(df_gabungan)} baris pekerjaan.")
    print(f"File master tersimpan sebagai: '{nama_file_master}'")
    
    # Tampilkan sedikit hasil akhirnya
    display(df_gabungan.tail())

[*] Memulai proses penggabungan file...
✅ Ditemukan 10 file untuk digabungkan:
   - raw_job_catalog_page_11_to_20.csv
   - raw_job_catalog_page_1_to_10.csv
   - raw_job_catalog_page_21_to_30.csv
   - raw_job_catalog_page_31_to_40.csv
   - raw_job_catalog_page_41_to_50.csv
   - raw_job_catalog_page_51_to_60.csv
   - raw_job_catalog_page_61_to_70.csv
   - raw_job_catalog_page_71_to_80.csv
   - raw_job_catalog_page_81_to_90.csv
   - raw_job_catalog_page_91_to_100.csv

🎉 PENGGABUNGAN SUKSES!
Total keseluruhan data yang berhasil kamu kumpulkan: 3025 baris pekerjaan.
File master tersimpan sebagai: 'MASTER_RAW_JOB_CATALOG.csv'


,Job Title,Job URL,Company,Location,Job Description,required_skills
3020,Senior Atoti Engineer,https://id.jobstreet.com/id/job/90846930?type=...,Macquarie Group,Indonesia,"Additional office locations | Melbourne, Sydne...","[""Java""]"
3021,Asia Equity Sales,https://id.jobstreet.com/id/job/90286794?type=...,Macquarie Group,Indonesia,Additional office locations | Taipei | Job ID ...,[]
3022,Product Manager (Mandarin Speaker),https://id.jobstreet.com/id/job/90773438?type=...,PT Apex Nusantara Indo,Jakarta Raya,About the Role | We are seeking a skilled and ...,"[""Figma"", ""UI/UX"", ""Data Analysis"", ""Agile"", ""..."
3023,"Attorney in Bukit, Indonesia (General)",https://id.jobstreet.com/id/job/91372361?type=...,In-House,Indonesia,"Job Title: Vice President, Digital Platform Go...",[]
3024,Sales Manager Mandarin Speaker - Technology So...,https://id.jobstreet.com/id/job/88221700?type=...,WireHire,Jakarta Raya,Job Title: Sales Manager Mandarin Speaker | Lo...,[]
